In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import time
import copy
import pickle

import numpy as np
import pandas as pd
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))


In [ ]:
def display_df(df):
    display(HTML(df.to_html()))
    return None

# Exclude SMILES

In [ ]:
df01 = pd.read_csv("/home/hai/data/Compound_Screen/AchE_ML_2306/data/process/pdXY_labeled_rdkit_descriptors_108ft_imputed_std.csv")
print("df01", df01.shape)
display_df(df01.head(1))
smiles01 = df01["smiles"].unique().tolist()
print("smiles01", len(smiles01))

df02 = pd.read_csv("/home/hai/data/Compound_Screen/BACE1/data/process/pdXY_labeled_rdkit_descriptors_103ft_imputed_std.csv")
print("df02", df02.shape)
display_df(df02.head(1))
smiles02 = df02["smiles"].unique().tolist()
print("smiles02", len(smiles02))

df03 = pd.read_csv("data/process/pdXY_ab42_train_test_rdkit_descriptors_109ft_imputed_std.csv")
print("df03", df03.shape)
display_df(df03.head(1))
smiles03 = df03["smiles"].unique().tolist()
print("smiles03", len(smiles03))

SMILES_EXCL = list(set(smiles01 + smiles02 + smiles03))
print("SMILES_EXCL", len(SMILES_EXCL))

## chembl36

In [ ]:
assert False
df04 = pd.read_csv("results/ensemble/chembl33_pred.csv")
print("df04",df04.shape)
display_df(df04.head(1))

df05 = pd.read_csv("results/ensemble/chembl36excl33_pred.csv")
print("df05",df05.shape)
display_df(df05.head(1))

cnames = ["new_id", "smiles", "pred_ens"]
df06 = pd.concat([df04[cnames], df05[cnames]], axis=0, ignore_index=True)
df06 = df06.rename(columns={"new_id": "chembl_id", "pred_ens": "ML_Abeta42_dG"})
df06 = df06.sort_values(by="ML_Abeta42_dG")
df06["database"] = "chembl36"
cnames = ["chembl_id", "database", "smiles", "ML_Abeta42_dG"]
df06 = df06[cnames]
print("df06",df06.shape)

ii = df06["smiles"].apply(lambda s: s.strip().lower()).isin([s.strip().lower() for s in SMILES_EXCL])
df06 = df06[~ii]
print("df06",df06.shape)

display_df(df06.head(1))
df06.to_csv("returns/ML_Abeta42_chembl36_20260320.csv", index=False)

## combine Abeta42, AchE, BACE1

In [ ]:
df00 = pd.read_csv("returns/ML_Abeta42_chembl36_20260320.csv")
print("df00", df00.shape)
display_df(df00.head(1))

# AchE
df01 = pd.read_csv("../AchE_ML_2306/results/xgb/chembl33_AChE_2306.csv")
print("df01", df01.shape)
display_df(df01.head(1))

df01a = pd.read_csv("../AchE_ML_2306/results/xgb/chembl36excl33_AChE_2306.csv")
print("df01a", df01a.shape)
display_df(df01a.head(1))

df01 = pd.concat([df01, df01a], axis=0, ignore_index=True)
df01 = df01[["smiles", "pred_AChE_2306"]].rename(columns={"pred_AChE_2306": "ML_AChE_dG"})
print("df01", df01.shape)
df01 = df01.drop_duplicates(subset=["smiles"])
print("df01", df01.shape)
del df01a
display_df(df01.head(1))



# BACE1
df02 = pd.read_csv("../BACE1/results/xgb/chembl33_BACE1.csv")
print("df02", df02.shape)
display_df(df02.head(1))

df02a = pd.read_csv("../BACE1/results/xgb/chembl36excl33_BACE1.csv")
print("df02a", df02a.shape)
display_df(df02a.head(1))

df02 = pd.concat([df02, df02a], axis=0, ignore_index=True)
df02 = df02[["smiles", "pred_BACE1"]].rename(columns={"pred_BACE1": "ML_BACE1_dG"})
print("df02", df02.shape)
df02 = df02.drop_duplicates(subset=["smiles"])
print("df02", df02.shape)
del df02a
display_df(df02.head(1))


In [ ]:
assert True
print("df00", df00.shape)
df03 = df00.merge(df01, how="left", on="smiles")
print("df03", df03.shape)
df03 = df03.merge(df02, how="left", on="smiles")
print("df03", df03.shape)
df03["final_score"] = df03["ML_Abeta42_dG"].apply(lambda x: (-x)**4) * df03["ML_AChE_dG"].apply(lambda x: -x) * df03["ML_BACE1_dG"].apply(lambda x: -x)
df03["final_score"] = df03["final_score"].apply(lambda x: x**(1/6))
df03 = df03.sort_values(by="final_score", ascending=False).reset_index(drop=True)
print("df03", df03.shape)
display_df(df03.head(1))

df03.to_csv("returns/ML_Abeta42_AchE-BACE1_chembl36.csv", index=False)

## amylovis201

In [ ]:
df04 = pd.read_csv("results/ensemble/amylovis201_pred.csv")
print("df04",df04.shape)
display_df(df04.head(1))

cnames = ["new_id", "smiles", "pred_ens"]
df06 = df04[cnames]
df06 = df06.rename(columns={"new_id": "chembl_id", "pred_ens": "ML_Abeta42_dG"})
df06 = df06.sort_values(by="ML_Abeta42_dG")
df06["database"] = "amylovis201"
cnames = ["chembl_id", "database", "smiles", "ML_Abeta42_dG"]
df06 = df06[cnames]
print("df06",df06.shape)

ii = df06["smiles"].apply(lambda s: s.strip().lower()).isin([s.strip().lower() for s in SMILES_EXCL])
df06 = df06[~ii]
print("df06",df06.shape)

display_df(df06.head(1))
df06.to_csv("returns/ML_Abeta42_amylovis201_20260320.csv", index=False)

## combine Abeta42, AchE, BACE1

In [ ]:
df00 = pd.read_csv("returns/ML_Abeta42_amylovis201_20260320.csv")
print("df00", df00.shape)
display_df(df00.head(1))

# AchE
df01 = pd.read_csv("../AchE_ML_2306/results/xgb/amylovis201_AChE_2306.csv")
print("df01", df01.shape)
df01 = df01[["smiles", "pred_AChE_2306"]].rename(columns={"pred_AChE_2306": "ML_AChE_dG"})
print("df01", df01.shape)
df01 = df01.drop_duplicates(subset=["smiles"])
print("df01", df01.shape)
display_df(df01.head(1))



# BACE1
df02 = pd.read_csv("../BACE1/results/xgb/amylovis201_BACE1.csv")
print("df02", df02.shape)

df02 = df02[["smiles", "pred_BACE1"]].rename(columns={"pred_BACE1": "ML_BACE1_dG"})
print("df02", df02.shape)
df02 = df02.drop_duplicates(subset=["smiles"])
print("df02", df02.shape)
display_df(df02.head(1))


In [ ]:
assert True
print("df00", df00.shape)
df03 = df00.merge(df01, how="left", on="smiles")
print("df03", df03.shape)
df03 = df03.merge(df02, how="left", on="smiles")
print("df03", df03.shape)

df03["final_score"] = df03["ML_Abeta42_dG"].apply(lambda x: (-x)**4) * df03["ML_AChE_dG"].apply(lambda x: -x) * df03["ML_BACE1_dG"].apply(lambda x: -x)
df03["final_score"] = df03["final_score"].apply(lambda x: x**(1/6))
df03 = df03.sort_values(by="final_score", ascending=False).reset_index(drop=True)
print("df03", df03.shape)
display_df(df03.head(1))

df03.to_csv("returns/ML_Abeta42_AchE-BACE1_amylovis201.csv", index=False)

In [ ]:
df03

## ref01

In [ ]:
df04 = pd.read_csv("results/ensemble/ref01_pred.csv")
print("df04",df04.shape)
display_df(df04.head(1))

cnames = ["new_id", "smiles", "pred_ens"]
df06 = df04[cnames]
df06 = df06.rename(columns={"new_id": "chembl_id", "pred_ens": "ML_Abeta42_dG"})
df06 = df06.sort_values(by="ML_Abeta42_dG")
df06["database"] = "ref01"
cnames = ["chembl_id", "database", "smiles", "ML_Abeta42_dG"]
df06 = df06[cnames]
print("df06",df06.shape)

ii = df06["smiles"].apply(lambda s: s.strip().lower()).isin([s.strip().lower() for s in SMILES_EXCL])
df06 = df06[~ii]
print("df06",df06.shape)

display_df(df06.head(1))
df06.to_csv("returns/ML_Abeta42_ref01_20260320.csv", index=False)

In [ ]:
df00 = pd.read_csv("returns/ML_Abeta42_ref01_20260320.csv")
print("df00", df00.shape)
display_df(df00.head(1))

# AchE
df01 = pd.read_csv("../AchE_ML_2306/results/xgb/ref01_AChE_2306.csv")
print("df01", df01.shape)
df01 = df01[["smiles", "pred_AChE_2306"]].rename(columns={"pred_AChE_2306": "ML_AChE_dG"})
print("df01", df01.shape)
df01 = df01.drop_duplicates(subset=["smiles"])
print("df01", df01.shape)
display_df(df01.head(1))



# BACE1
df02 = pd.read_csv("../BACE1/results/xgb/ref01_BACE1.csv")
print("df02", df02.shape)

df02 = df02[["smiles", "pred_BACE1"]].rename(columns={"pred_BACE1": "ML_BACE1_dG"})
print("df02", df02.shape)
df02 = df02.drop_duplicates(subset=["smiles"])
print("df02", df02.shape)
display_df(df02.head(1))


In [ ]:
assert True
print("df00", df00.shape)
df03 = df00.merge(df01, how="left", on="smiles")
print("df03", df03.shape)
df03 = df03.merge(df02, how="left", on="smiles")
print("df03", df03.shape)

df03["final_score"] = df03["ML_Abeta42_dG"].apply(lambda x: (-x)**4) * df03["ML_AChE_dG"].apply(lambda x: -x) * df03["ML_BACE1_dG"].apply(lambda x: -x)
df03["final_score"] = df03["final_score"].apply(lambda x: x**(1/6))
#df03 = df03.sort_values(by="final_score", ascending=False).reset_index(drop=True)
print("df03", df03.shape)
display_df(df03.head())

df03.to_csv("returns/ML_Abeta42_AchE-BACE1_ref01.csv", index=False)

# Nafosted 2603

## Cong viec 1.2, chi dau phan tu

In [ ]:
df00 = pd.read_csv("data/process/pdXY_ab42_train_test_rdkit_descriptors_200ft.csv")
print(df00.shape)
display_df(df00.head(1))

cnames_exld = ["new_id", "code", "smiles_len", "train_test"]
cnames = [c for c in df00.columns if c not in cnames_exld]
cnames = ["smiles", "dG"] + [c for c in cnames if c not in ["smiles", "dG"] ]
df00 = df00[cnames]
print(df00.shape)
display_df(df00.head(1))
df00.to_csv("returns/nafosted2603/congviec_1.2_chi_dau_phan_tu.csv", index=False)

## Cong viec 2.2 train_test

In [ ]:
df00 = pd.read_csv("data/process/pdXY_ab42_train_test_rdkit_descriptors_200ft.csv")
print(df00.shape)
display_df(df00.head(1))

cnames = ["smiles", "train_test"]
df00 = df00[cnames]
print(df00.shape)
display_df(df00.head(1))
df00.to_csv("returns/nafosted2603/congviec_2.2_train_test.csv", index=False)

In [ ]:
## Cong viec 2.3 